In [1]:
import csv
import re

import sqlite3
from collections import Counter

In [2]:
conn = sqlite3.connect('data.db')
cursor = conn.cursor()

In [3]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS metadata (
        id      INTEGER NOT NULL PRIMARY KEY,
        title   TEXT NOT NULL,
        year    INTEGER
    );
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS charcount (
        work    INTEGER NOT NULL,
        char    INTEGER NOT NULL,
        count   INTEGER NOT NULL DEFAULT 0,
        PRIMARY KEY (work, char),
        FOREIGN KEY (work) REFERENCES metadata(id)
    );
''')

In [4]:
def int_or_null(x: str):
    try:
        return int(x)
    except ValueError:
        return 'NULL'

In [5]:
work_ids = set()

with open('data/meta_info.csv', 'r', encoding='utf-8') as metafile:
    lines = csv.DictReader(metafile)

    for line in lines:
        work_id = int(line['作品ID'])
        if work_id in work_ids:
            continue

        work_ids.add(work_id)

        query = f'''
            INSERT INTO metadata VALUES (
                {work_id},
                "{line['作品名'].strip()}",
                {int_or_null(re.split('\\D', line['底本初版発行年1'], 1)[0])}
            );
        '''

        try:
            cursor.execute(query)
        except (sqlite3.IntegrityError, sqlite3.OperationalError) as e:
            print(query)
        

In [6]:
work_ids = set()

with open('data/main_text.csv', 'r', encoding='utf-8') as textfile:
    lines = textfile.readlines()

    for line in lines[1:]:
        work_id, text = line.split(',', 1)
        counts = Counter(text.strip())

        if work_id in work_ids:
            continue

        work_ids.add(work_id)

        # ','.join(f'({int(work)}, "{k}", {v})' for k, v in counts.items())

        for char, count in counts.items():
            query = f'''
                INSERT INTO charcount VALUES ({int(work_id)}, {ord(char)}, {count});
            '''

            try:
                cursor.execute(query)
            except (sqlite3.IntegrityError, sqlite3.OperationalError) as e:
                print(query)


In [7]:
conn.commit()
conn.close()